## GRPO after SFT Training Algorithm on Llama-3.2-1B-Instruct

### Environment setup

In [ ]:
# Environment Setup & Repository Cloning

# Change working directory to Kaggle's writable storage
import os
os.chdir('/kaggle/working')

# Remove any previous clone
!rm -rf /kaggle/working/Emergence-of-Thinking

# Clone the official paper repository
!git clone https://github.com/GuanghaoYe/Emergence-of-Thinking.git

# Move into the cloned repository
os.chdir('/kaggle/working/Emergence-of-Thinking')

# Fix Ray version from 2.12.0 → 2.31.0 in all config files
print('Fixing Ray version references')
for root, dirs, files in os.walk('.'):
    if '.git' in root:
        continue
    for file in files:
        if file.endswith(('.txt', '.py', '.toml')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r') as f:
                    content = f.read()
                if 'ray' in content.lower() and '2.12.0' in content:
                    with open(filepath, 'w') as f:
                        f.write(content.replace('2.12.0', '2.31.0'))
                    print(f'  Fixed: {filepath}')
            except:
                pass


In [ ]:
# Install Dependencies (Kaggle-optimized, with fixes)

!pip install --upgrade pip setuptools wheel --user -q

!pip install torch torchvision torchaudio --user -q

!pip install numpy cython packaging --user -q

# Remove flash-attn from requirements.txt to avoid build failure
print('Removing flash-attn from requirements.txt')
req_file = '/kaggle/working/Emergence-of-Thinking/requirements.txt'
if os.path.exists(req_file):
    with open(req_file, 'r') as f:
        lines = f.readlines()
    with open(req_file, 'w') as f:
        for line in lines:
            if 'flash-attn' not in line.lower():
                f.write(line)
    print('Removed flash-attn from requirements.txt')

# Skip flash-attn build
os.environ['FLASH_ATTENTION_SKIP_CUDA_BUILD'] = 'TRUE'

# Try installing the core project (without flash-attn dependency)
print('Attempting editable install')
!pip install -e . --user -q

# If editable fails, try regular install
if os.system('pip show emergence-of-thinking > /dev/null 2>&1') != 0:
    print('Editable install failed. Trying regular install')
    !pip install . --user -q

# Add repo to Python path
import sys
sys.path.insert(0, '/kaggle/working/Emergence-of-Thinking')
print("Setup complete. Repository added to Python path.")

In [ ]:
!pip install transformers==4.46.3 accelerate --user -q


The following script is used for resumability reasons. More specifically, when a Kaggle session ends or disconnects, the script ensures that if the user uploads the checkpoint files (.csv and .pkl) as a Dataset, they are placed in the correct working directories.

In [ ]:
import os
import shutil

# Dataset path
input_dataset_dir = "/kaggle/input/datasets/kaggle_username/kaggle_dataset"  

working_dir = "/kaggle/working"
sub_output_dir = os.path.join(working_dir, "grpo_length_reward_gradientaccumulationplusnoboxpenalty")


# Create the desiredd subfolders inside /kaggle/working
os.makedirs(sub_output_dir, exist_ok=True)
print("Target subdirectories verified inside /kaggle/working")

# Copy Checkpoint into the root of /kaggle/working
src_checkpoint = os.path.join(input_dataset_dir, "grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl")
dst_checkpoint = os.path.join(working_dir, "grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl")

if os.path.exists(src_checkpoint):
    shutil.copy(src_checkpoint, dst_checkpoint)
    print("Checkpoint perfectly placed at: /kaggle/working/grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl")
else:
    print(f"Looking for: {src_checkpoint}")
    print("Could not find 'grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl' in that specific directory. Check file spelling inside the dataset.")

#  Copy length log into the subfolder
src_log_csv = os.path.join(input_dataset_dir, "length_log.csv")
src_log_raw = os.path.join(input_dataset_dir, "length_log")
dst_log = os.path.join(sub_output_dir, "length_log.csv")

if os.path.exists(src_log_csv):
    shutil.copy(src_log_csv, dst_log)
    print("Metrics log perfectly placed inside subfolder at: /kaggle/working/grpo_length_reward_gradientaccumulationplusnoboxpenalty/length_log.csv")
elif os.path.exists(src_log_raw):
    shutil.copy(src_log_raw, dst_log)
    print("Metrics log (renamed from raw) perfectly placed inside subfolder at: /kaggle/working/grpo_length_reward_gradientaccumulationplusnoboxpenalty/length_log.csv")
else:
    print(f"Could not find 'length_log.csv' or 'length_log' at: {input_dataset_dir}")

#### Model Loading

In order to load the model Llama-3.2-1B-Instruct from Hugging Face, one must have been granted access to the model on Hugging Face and attach the Hugging Face token as a Kaggle Secret.

In [ ]:
# Load Llama 3.2 1B Model with HF Token (Kaggle Secrets)

import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient

# Retrieve HF token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Set cache directories
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/hf_cache'
os.makedirs('/kaggle/working/hf_cache', exist_ok=True)

model_name = "meta-llama/Llama-3.2-1B-Instruct"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)
print("Model loaded successfully!")

# Quick inference test
prompt = "What is 17 + 25? Let's think step by step."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model Response:")
print(response)

In [ ]:
import os
import json

# Verify training data
data_path = '/kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl'
print('Training data exists:', os.path.exists(data_path))

if os.path.exists(data_path):
    with open(data_path, 'r') as f:
        sample = json.loads(f.readline())
    print('Sample keys:', list(sample.keys()))
    print('Sample input (first 200 chars):', sample.get('input', '')[:200])
else:
    print('WARNING: Training data not found. Check the repo structure.')

# Verify evaluation data
eval_data = '/kaggle/working/Emergence-of-Thinking/evaluation/data/math'
print('Eval data exists:', os.path.exists(eval_data))

# Create necessary directories (Kaggle paths)
os.makedirs('/kaggle/working/tmp_code', exist_ok=True)
os.makedirs('/kaggle/working/Emergence-of-Thinking/logs/openrlhf_train_ppo', exist_ok=True)
print('Directories ready.')

In [ ]:
import shutil, os

# Remove previous checkpoint directory if it exists (clean start)
checkpoint_dir = '/kaggle/working/checkpoints'
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir, ignore_errors=True)
    print(f'Removed {checkpoint_dir}')
else:
    print(f'No existing {checkpoint_dir} to remove.')

# Remove Ray temporary directory
ray_tmp = '/tmp/ray'
if os.path.exists(ray_tmp):
    shutil.rmtree(ray_tmp, ignore_errors=True)
    print(f'Removed {ray_tmp}')
else:
    print(f'No existing {ray_tmp} to remove.')

# Check disk usage for Kaggle's working directory
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'/kaggle/working -- Free: {free/1e9:.1f} GB')

# Check /tmp
total, used, free = shutil.disk_usage('/tmp')
print(f'/tmp -- Free: {free/1e9:.1f} GB')

print('Ready for training')

In [ ]:
import os

# Path to deepspeed.py inside the cloned repo
filepath = '/kaggle/working/Emergence-of-Thinking/openrlhf/utils/deepspeed/deepspeed.py'

if os.path.exists(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    patched = False
    new_lines = []
    for line in lines:
        if 'assert state_dict_keys.issubset(' in line and not patched:
            indent = len(line) - len(line.lstrip())
            new_lines.append(' ' * indent + 'state_dict_keys -= {"base_model.model.lm_head.weight"}\n')
            patched = True
        new_lines.append(line)

    if patched:
        with open(filepath, 'w') as f:
            f.writelines(new_lines)
        print('DeepSpeed patched successfully')
    else:
        print('Already patched or pattern not found')
else:
    print(f'File not found: {filepath}')

### GRPO

In [ ]:
!pip install peft datasets latex2sympy2 -q

In [ ]:
%%writefile /kaggle/working/Emergence-of-Thinking/train_sft_local.py
import torch
import json
import argparse
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ['HF_TOKEN'] = hf_token

def clean_text(text):
    if not text:
        return ""
    return re.sub(r'\W+', '', text.lower().strip())

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.2-1B-Instruct")
    # Χρήση του επίσημου Hugging Face dataset ID
    parser.add_argument("--dataset_name", type=str, default="gghfez/QwQ-LongCoT-130K-cleaned")
    parser.add_argument("--output_dir", type=str, default="/kaggle/working/sft_math_local")
    parser.add_argument("--max_samples", type=int, default=2000)
    parser.add_argument("--batch_size", type=int, default=2)
    parser.add_argument("--grad_accum", type=int, default=4)
    parser.add_argument("--epochs", type=int, default=3)   
    parser.add_argument("--lr", type=float, default=2e-5)
    args = parser.parse_args()

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    tokenizer.pad_token = tokenizer.eos_token

    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.bfloat16,
        device_map={"": local_rank} 
    )
    
    # LoRA Config
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

    # Loading of MATH500 to secure no data leakage
    print("loading MATH500 for data leakage checking...")
    math500 = load_dataset("HuggingFaceH4/MATH-500", split="test", token=hf_token)
    math500_problems = {clean_text(ex["problem"]) for ex in math500 if ex.get("problem")}

    # Streaming the sft dataset
    print(f"Streaming of dataset: {args.dataset_name}...")
    raw_stream = load_dataset(args.dataset_name, split="train", streaming=True, token=hf_token)

    examples = []
    leakage_count = 0

    for item in raw_stream:
        if len(examples) >= args.max_samples:
            break
            
        if "messages" in item:
            user_prompt = next((m["content"] for m in item["messages"] if m["role"] == "user"), "")
            assistant_response = next((m["content"] for m in item["messages"] if m["role"] == "assistant"), "")
        else:
            user_prompt = item.get("query", item.get("instruction", item.get("input", "")))
            assistant_response = item.get("response", item.get("output", item.get("solution", "")))

        # Check for Data Leakage
        if clean_text(user_prompt) in math500_problems:
            leakage_count += 1
            continue  # Παράλειψη αυτού του δείγματος

        full_text = f"Question: {user_prompt}\nReasoning and Solution:\n{assistant_response}{tokenizer.eos_token}"
        examples.append({"text": full_text})

    print(f"Found {len(examples)} pure samples.")
    print(f"Discarded {leakage_count} due to data leakage.")

    dataset = Dataset.from_list(examples)

    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=2048)
        
    tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    training_args = TrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        num_train_epochs=args.epochs,
        logging_steps=10,
        save_steps=250,
        bf16=True,
        report_to="none"
    )
    
    trainer = Trainer(model=model, args=training_args, train_dataset=tokenized, data_collator=data_collator)
    trainer.train()
    
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

if __name__ == "__main__":
    main()


**Install Dependencies**

In [ ]:
# ==========================================
# Local Environment Installation Cell
# ==========================================

# 1. Install core dependencies, API routers, evaluation utilities, and the exact omegaconf version
!pip install torch transformers fastapi uvicorn pebble timeout_decorator word2number latex2sympy2 "omegaconf==2.4.0.dev3"

# 2. Install the repository's base requirements
!pip install -r requirements.txt 

# 3. Install Flash Attention (isolated to its own line so the flag only applies here)
!pip install flash-attn --no-build-isolation

# 4. vLLM is commented out for your local execution
# !pip install vllm==0.6.4.post1

In [ ]:
# !cd /kaggle/working/Emergence-of-Thinking && python train_sft_local.py \
!cd /kaggle/working/Emergence-of-Thinking && accelerate launch --multi_gpu --num_processes 2 train_sft_local.py \
    --model_name meta-llama/Llama-3.2-1B-Instruct \
    --dataset_name gghfez/QwQ-LongCoT-130K-cleaned \
    --max_samples 2000 \
    --batch_size 2 \
    --grad_accum 4 \
    --epochs 2 \
    --output_dir /kaggle/working/sft_math_local